In [1]:
import polars as pl
import json

In [2]:
path_to_card = "../temp/data/raw/card.20250819.json"

with open(path_to_card, "r") as f:
    card_data: dict[str, dict] = json.load(f)

In [4]:
keys_to_pop = ['_version', '_comment', '_timestamp']
for key in keys_to_pop:
    card_data.pop(key, None)

In [5]:
# Get values for specific keys
data_rows = []
for record in card_data.values():
    protein_sequence = ""
    dna_sequence = ""
    antibiotics = []
    drug_classes = []
    resistance_mechanisms = []
    amr_gene_families = []

    try:
        for value in record["model_sequences"]["sequence"].values():
            protein_sequence = value["protein_sequence"]["sequence"]
            dna_sequence = value["dna_sequence"]["sequence"]

        for value in record["ARO_category"].values():
            category_name = value["category_aro_name"]
            category_class_name = value["category_aro_class_name"]
            if category_class_name == "Antibiotic":
                antibiotics.append(category_name)
            elif category_class_name == "Drug Class":
                drug_classes.append(category_name)
            elif category_class_name == "Resistance Mechanism":
                resistance_mechanisms.append(category_name)
            elif category_class_name == "AMR Gene Family":
                amr_gene_families.append(category_name)
    except KeyError:
        print(f"Key not found in record {record["model_id"]}, skipping...")
        continue

    data_rows.append(
        {
            "protein_sequence": protein_sequence,
            "dna_sequence": dna_sequence,
            "antibiotics": antibiotics,
            "drug_classes": drug_classes,
            "resistance_mechanisms": resistance_mechanisms,
            "amr_gene_families": amr_gene_families,
        }
    )

Key not found in record 2176, skipping...
Key not found in record 2177, skipping...
Key not found in record 2178, skipping...
Key not found in record 2179, skipping...
Key not found in record 2180, skipping...
Key not found in record 2181, skipping...
Key not found in record 2182, skipping...
Key not found in record 2183, skipping...
Key not found in record 2184, skipping...
Key not found in record 2185, skipping...
Key not found in record 2186, skipping...
Key not found in record 2551, skipping...
Key not found in record 2705, skipping...
Key not found in record 2707, skipping...
Key not found in record 2695, skipping...
Key not found in record 2712, skipping...
Key not found in record 2724, skipping...
Key not found in record 2729, skipping...
Key not found in record 2731, skipping...
Key not found in record 2732, skipping...
Key not found in record 2733, skipping...
Key not found in record 2745, skipping...
Key not found in record 2748, skipping...
Key not found in record 2760, skip

In [6]:
print(f"Extracted {len(data_rows)} records from CARD data.")

Extracted 6404 records from CARD data.


In [7]:
df = pl.DataFrame(data_rows)

In [8]:
df

protein_sequence,dna_sequence,antibiotics,drug_classes,resistance_mechanisms,amr_gene_families
str,str,list[str],list[str],list[str],list[str]
"""MKAYFIAILTLFTCIATVVRAQQMSELENR…","""ATGAAAGCATATTTCATCGCCATACTTACC…","[""cephaloridine""]","[""cephalosporin""]","[""antibiotic inactivation""]","[""CblA beta-lactamase""]"
"""MRYIRLCIISLLAALPLAVHASPQPLEQIK…","""ATGCGTTATATTCGCCTGTGTATTATCTCC…",[],"[""cephalosporin"", ""penicillin beta-lactam""]","[""antibiotic inactivation""]","[""SHV beta-lactamase""]"
"""MIGLIVARSKNNVIGKNGNIPWKIKGEQKQ…","""ATGATAGGTTTGATTGTTGCGAGGTCAAAG…","[""trimethoprim""]","[""diaminopyrimidine antibiotic""]","[""antibiotic target replacement""]","[""trimethoprim resistant dihydrofolate reductase dfr""]"
"""MVTKRVQRMMFAAAACIPLLLGSAPLYAQT…","""ATGGTGACAAAGAGAGTGCAACGGATGATG…",[],"[""cephalosporin""]","[""antibiotic inactivation""]","[""CTX-M beta-lactamase""]"
"""MELPNIMHPVAKLSTALAAALMLSGCMPGE…","""ATGGAATTGCCCAATATTATGCACCCGGTC…",[],"[""carbapenem"", ""cephalosporin"", ""penicillin beta-lactam""]","[""antibiotic inactivation""]","[""NDM beta-lactamase""]"
…,…,…,…,…,…
"""MKAAAKTQKPKRQEEHANFISWRFALLCGC…","""ATGAAAGCAGCGGCGAAAACGCAGAAACCA…","[""ceftazidime"", ""aztreonam"", ""ceftaroline""]","[""monobactam"", ""cephalosporin"", ""penicillin beta-lactam""]","[""antibiotic target alteration""]","[""Penicillin-binding protein mutations conferring resistance to beta-lactam antibiotics""]"
"""MLKSSWRKTALMAAAAVPLLLASGSLWASA…","""ATGTTGAAAAGTTCGTGGCGTAAAACCGCC…",[],"[""monobactam"", ""cephalosporin"", ""penicillin beta-lactam""]","[""antibiotic inactivation""]","[""OXY beta-lactamase""]"
"""MSIIATVKIGPDEISAMRAVLDLFGKEFED…","""ATGAGCATCATTGCAACCGTCAAGATCGGC…","[""astromicin"", ""amikacin"", … ""gentamicin""]","[""aminoglycoside antibiotic""]","[""antibiotic inactivation""]","[""aminoglycoside bifunctional resistance protein""]"


In [9]:
# Put it in ../temp/data/interim/card_extracted.parquet
df.write_parquet("../temp/data/interim/card_extracted.parquet")

In [10]:
# columns with empty lists in 'drug_classes'
empty_drug_class_count = df.filter(pl.col("drug_classes").list.len() == 0).height
print(f"Number of records with empty 'drug_classes': {empty_drug_class_count}")

Number of records with empty 'drug_classes': 7


In [11]:
# Drop those rows
df = df.filter(pl.col("drug_classes").list.len() > 0)

In [20]:
# Extract drug classes, and one hot encode them for future analysis

# explode and pivot
drug_class_vocab = (
    df.select(pl.col("drug_classes").explode())
      .unique()
      .sort("drug_classes")
      .to_series()
      .to_list()
)

In [27]:
df_one_hot = df.with_columns(
    [
        pl.col("drug_classes")
        .list.contains(drug_class)
        .cast(pl.Int8)
        .alias(drug_class)
        for drug_class in drug_class_vocab
    ]
)

In [28]:
# ../temp/data/interim/card_amr.parquet
df_one_hot.write_parquet("../temp/data/interim/card_amr.parquet")